# 📈 View — Receita Mensal

Validação da view `vw_receita_mensal` antes de mover para o Streamlit.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from views.vw_receita_mensal import get_receita_mensal

pd.set_option('display.float_format', '{:.2f}'.format)

pedidos    = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
pagamentos = pd.read_csv("../dados/pagamentos_limpo.csv")
clientes = pd.read_csv("../dados/clientes_limpo.csv")

print("Dados carregados!")

Dados carregados!


## 🧪 Testando o código antes de criar a view

In [2]:
# =========================================================
# JOIN ENTRE PEDIDOS, PAGAMENTOS E CLIENTES
# =========================================================

df = (
    pedidos
    .merge(
        pagamentos,
        on='order_id',
        how='left'
    )
    .merge(
        clientes[['customer_id', 'customer_state']],
        on='customer_id',
        how='left'
    )
)

# Filtrando somente pedidos entregues
df = df[df['order_status'] == 'delivered']

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.00,credit_card,1.00,18.12,SP
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,3.00,voucher,1.00,2.00,SP
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2.00,voucher,1.00,18.59,SP
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.00,boleto,1.00,141.46,BA
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.00,credit_card,3.00,179.12,GO


In [3]:
# Extraindo ano e mês

df['ano'] = df['order_purchase_timestamp'].dt.year
df['mes'] = df['order_purchase_timestamp'].dt.month
df['mes_nome'] = df['order_purchase_timestamp'].dt.strftime('%b')
df['ano_mes'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)


# Tradução do nome do mês

meses_traducao = {
    'Jan': 'Jan', 'Feb': 'Fev', 'Mar': 'Mar',
    'Apr': 'Abr', 'May': 'Mai', 'Jun': 'Jun',
    'Jul': 'Jul', 'Aug': 'Ago', 'Sep': 'Set',
    'Oct': 'Out', 'Nov': 'Nov', 'Dec': 'Dez'
}

df['mes_nome'] = df['mes_nome'].map(meses_traducao)


# =========================================================
# AGRUPAMENTO DA RECEITA MENSAL POR ESTADO
# =========================================================

receita_mensal = (
    df.groupby([
        'customer_state',
        'ano',
        'mes',
        'mes_nome',
        'ano_mes'
    ])
    .agg(
        total_pedidos=('order_id', 'nunique'),
        receita_total=('payment_value', 'sum')
    )
    .reset_index()
    .sort_values(['customer_state', 'ano', 'mes'])
)

receita_mensal['receita_total'] = (
    receita_mensal['receita_total'].round(2)
)

receita_mensal.head(15)

,customer_state,ano,mes,mes_nome,ano_mes,total_pedidos,receita_total
0,AC,2017,1,Jan,2017-01,2,723.15
1,AC,2017,2,Fev,2017-02,3,597.40
2,AC,2017,3,Mar,2017-03,2,530.18
3,AC,2017,4,Abr,2017-04,5,1351.51
4,AC,2017,5,Mai,2017-05,8,2382.64
5,AC,2017,6,Jun,2017-06,4,510.27
6,AC,2017,7,Jul,2017-07,5,794.40
7,AC,2017,8,Ago,2017-08,4,765.83
8,AC,2017,9,Set,2017-09,5,2029.23
9,AC,2017,10,Out,2017-10,5,863.80


In [4]:
# =========================================================
# CALENDÁRIO MENSAL COMPLETO POR ESTADO
# =========================================================
#
# O groupby original só cria linhas para meses que possuem
# movimentação.
#
# Aqui criamos todas as combinações de:
#
#   Estado × Mês
#
# Os meses sem movimentação recebem receita = 0.
# =========================================================

estados = receita_mensal['customer_state'].unique()

data_inicio = receita_mensal['ano_mes'].min()
data_fim = receita_mensal['ano_mes'].max()

calendario = pd.date_range(
    start=data_inicio,
    end=data_fim,
    freq='MS'
).to_period('M').astype(str)

calendario_completo = pd.MultiIndex.from_product(
    [estados, calendario],
    names=['customer_state', 'ano_mes']
).to_frame(index=False)

# Criando ano e mês a partir do calendário
calendario_completo['ano'] = (
    calendario_completo['ano_mes']
    .str[:4]
    .astype(int)
)

calendario_completo['mes'] = (
    calendario_completo['ano_mes']
    .str[5:7]
    .astype(int)
)

# Nome do mês
calendario_completo['mes_nome'] = (
    pd.to_datetime(calendario_completo['ano_mes'])
    .dt.strftime('%b')
    .map(meses_traducao)
)

# =========================================================
# JUNTAR O CALENDÁRIO COM A RECEITA
# =========================================================

receita_mensal = calendario_completo.merge(
    receita_mensal,
    on=['customer_state', 'ano_mes', 'ano', 'mes', 'mes_nome'],
    how='left'
)

# Meses sem movimentação recebem zero
receita_mensal['total_pedidos'] = (
    receita_mensal['total_pedidos']
    .fillna(0)
)

receita_mensal['receita_total'] = (
    receita_mensal['receita_total']
    .fillna(0)
)

# Ordenação cronológica dentro de cada estado
receita_mensal = (
    receita_mensal
    .sort_values(['customer_state', 'ano', 'mes'])
    .reset_index(drop=True)
)

receita_mensal.head(15)

,customer_state,ano_mes,ano,mes,mes_nome,total_pedidos,receita_total
0,AC,2016-09,2016,9,Set,0.00,0.00
1,AC,2016-10,2016,10,Out,0.00,0.00
2,AC,2016-11,2016,11,Nov,0.00,0.00
3,AC,2016-12,2016,12,Dez,0.00,0.00
4,AC,2017-01,2017,1,Jan,2.00,723.15
5,AC,2017-02,2017,2,Fev,3.00,597.40
6,AC,2017-03,2017,3,Mar,2.00,530.18
7,AC,2017-04,2017,4,Abr,5.00,1351.51
8,AC,2017-05,2017,5,Mai,8.00,2382.64
9,AC,2017-06,2017,6,Jun,4.00,510.27


In [5]:
# =========================================================
# RECEITA DO MÊS ANTERIOR
# =========================================================
#
# A view possui granularidade por estado e mês.
#
# Portanto, o shift deve ser feito separadamente
# para cada estado.
#
# Assim, um estado nunca pega a receita de outro estado.
# =========================================================

receita_mensal = (
    receita_mensal
    .sort_values(['customer_state', 'ano', 'mes'])
    .reset_index(drop=True)
)

receita_mensal['receita_mes_anterior'] = (
    receita_mensal
    .groupby('customer_state')['receita_total']
    .shift(1)
)

receita_mensal.head(15)

,customer_state,ano_mes,ano,mes,mes_nome,total_pedidos,receita_total,receita_mes_anterior
0,AC,2016-09,2016,9,Set,0.00,0.00,NaN
1,AC,2016-10,2016,10,Out,0.00,0.00,0.00
2,AC,2016-11,2016,11,Nov,0.00,0.00,0.00
3,AC,2016-12,2016,12,Dez,0.00,0.00,0.00
4,AC,2017-01,2017,1,Jan,2.00,723.15,0.00
5,AC,2017-02,2017,2,Fev,3.00,597.40,723.15
6,AC,2017-03,2017,3,Mar,2.00,530.18,597.40
7,AC,2017-04,2017,4,Abr,5.00,1351.51,530.18
8,AC,2017-05,2017,5,Mai,8.00,2382.64,1351.51
9,AC,2017-06,2017,6,Jun,4.00,510.27,2382.64


In [6]:
# =========================================================
# CÁLCULO DO MoM
# =========================================================

receita_mensal['variacao_mom_pct'] = (
    (
        receita_mensal['receita_total']
        - receita_mensal['receita_mes_anterior']
    )
    / receita_mensal['receita_mes_anterior']
    * 100
).round(2)

receita_mensal.loc[
    receita_mensal['receita_mes_anterior'] == 0,
    'variacao_mom_pct'
] = np.nan

In [8]:
receita_mensal[
    receita_mensal['customer_state'] == 'SP'
][[
    'customer_state',
    'ano_mes',
    'receita_total',
    'receita_mes_anterior'
]].head(15)

,customer_state,ano_mes,receita_total,receita_mes_anterior
600,SP,2016-09,0.00,NaN
601,SP,2016-10,13559.93,0.00
602,SP,2016-11,0.00,13559.93
603,SP,2016-12,0.00,0.00
604,SP,2017-01,44838.14,0.00
605,SP,2017-02,84444.00,44838.14
606,SP,2017-03,147665.99,84444.00
607,SP,2017-04,138822.04,147665.99
608,SP,2017-05,195065.48,138822.04
609,SP,2017-06,191008.60,195065.48


In [9]:
receita_mensal[
    receita_mensal['customer_state'] == 'SP'
][[
    'customer_state',
    'ano_mes',
    'receita_total',
    'receita_mes_anterior',
    'variacao_mom_pct'
]].head(15)

,customer_state,ano_mes,receita_total,receita_mes_anterior,variacao_mom_pct
600,SP,2016-09,0.00,NaN,NaN
601,SP,2016-10,13559.93,0.00,NaN
602,SP,2016-11,0.00,13559.93,-100.00
603,SP,2016-12,0.00,0.00,NaN
604,SP,2017-01,44838.14,0.00,NaN
605,SP,2017-02,84444.00,44838.14,88.33
606,SP,2017-03,147665.99,84444.00,74.87
607,SP,2017-04,138822.04,147665.99,-5.99
608,SP,2017-05,195065.48,138822.04,40.51
609,SP,2017-06,191008.60,195065.48,-2.08


In [10]:
df_receita = get_receita_mensal(
    pedidos,
    pagamentos,
    clientes
)

In [13]:
df_receita.head(2)

,customer_state,ano,mes,mes_nome,ano_mes,total_pedidos,receita_total,receita_mes_anterior,variacao_mom_pct
0,AC,2016,9,Set,2016-09,0,0.00,NaN,NaN
1,AC,2016,10,Out,2016-10,0,0.00,0.00,NaN


In [15]:
df_receita[
    df_receita['customer_state'] == 'SP'
][[
    'customer_state',
    'ano_mes',
    'receita_total',
    'receita_mes_anterior',
    'variacao_mom_pct'
]].head(15)

,customer_state,ano_mes,receita_total,receita_mes_anterior,variacao_mom_pct
600,SP,2016-09,0.00,NaN,NaN
601,SP,2016-10,13559.93,0.00,NaN
602,SP,2016-11,0.00,13559.93,NaN
603,SP,2016-12,0.00,0.00,NaN
604,SP,2017-01,44838.14,0.00,NaN
605,SP,2017-02,84444.00,44838.14,88.33
606,SP,2017-03,147665.99,84444.00,74.87
607,SP,2017-04,138822.04,147665.99,-5.99
608,SP,2017-05,195065.48,138822.04,40.51
609,SP,2017-06,191008.60,195065.48,-2.08
